# Testing the draft 2D tile plot (`team/new_graphics/tile.py`)

This notebook exercises `make_tile()`, the draft implementation of the new 2D
tile plot type. It is **not** wired into `.plot()` yet — this is for visually
checking the draft before integration.

The tile plot is for **two discrete variables**: each axis gets one labeled cell
per distinct value, and the cell color shows how often that (x, y) pair occurred.
This is the one 2D view a histogram can't give — every discrete value keeps its
own cell instead of being binned. Continuous 2D data is routed to `hist2d` /
`density2d` instead.

**How to run:** open this notebook from the `team/new_graphics/new_graphics_demos/`
folder using the `symbulate` conda environment, then Run All.

**What to look for in every plot:**
- A grid of filled cells, one per distinct (x, y) pair, in **viridis** (the
  package's sequential colormap, from `symbulate.mplstyle` — no hardcoded `Blues`)
- Colorbar on the right, labeled "Relative Frequency" (or "Count" when
  `normalize=False`), placed with `make_axes_locatable`
- Color scale starts at 0, so a pair that never occurred reads as "no data"
- Each axis tick labeled with the distinct value; axis labels read "X" and "Y"
- Title reads "Tile Plot"; no reference grid (the mesh covers the axes)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# This notebook lives in <repo>/team/new_graphics/new_graphics_demos/.
REPO_ROOT = Path.cwd().parents[2]

# Make `import symbulate` find the dev copy in this repo (not an older
# installed copy in site-packages), and `from tile import ...` find
# the draft module one folder up.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(Path.cwd().parent))

# The draft helper and its overlay-warning text
from tile import make_tile, TILE_OVERLAY_WARNING

# Apply the global style file (viridis colormap, fonts, spines).
# Not yet wired into the package, so we apply it by hand here.
STYLE_PATH = REPO_ROOT / "symbulate" / "symbulate.mplstyle"
plt.style.use(STYLE_PATH)

rng = np.random.default_rng(7)

## 1. Two correlated discrete variables

`x` is Poisson(3); `y` tracks `x` plus a little integer noise. The bright cells
should fall along a diagonal band — the joint distribution read as a table of
shaded relative frequencies, one cell per (x, y) integer pair, brightest where
pairs occur most and dark purple where a pair never occurred.

In [ ]:
x = rng.poisson(3, 4000)
y = np.clip(x + rng.integers(-2, 3, 4000), 0, None)

ax = plt.gca()
make_tile(x, y, ax)
plt.show()  # title, axis labels, and colorbar are set by make_tile

## 2. Counts — `normalize=False`

The same picture as raw counts instead of relative frequencies; the colorbar
label switches to "Count" and the cell values sum to the number of simulations.

In [ ]:
ax = plt.gca()
make_tile(x, y, ax, normalize=False)
plt.show()

## 3. Overlay behavior — the readability *warning* category

Per the overlay policy, two tile plots on the same axes is one of the designated
**warning** cases: the second tile still draws (overlay is never blocked outside
the `'marginal'` hard-error case), but its color scale competes with the first,
so a student-friendly warning prints below the plot.

The exact warning wording is still an open decision in `DECISIONS.md` — the text
printed here is the draft wording from `tile.py`.

In [ ]:
ax = plt.gca()
make_tile(x, y, ax)
make_tile(x + 2, y, ax)  # draws, and prints the warning
plt.show()

## 4. With real symbulate simulation data

The same plot, but the data comes from an actual `RV(...).sim(...)` call instead
of raw numpy: two independent Poisson variables, exactly the discrete × discrete
case the lookup table will route to a tile plot.

Caveat while testing today: creating `RVResults` currently calls `init_color()`,
which resets the color cycle to tab10, and importing `symbulate` switches the
matplotlib style back to the old stylesheet (both Phase 2 items) — so we re-apply
the style *after* importing and simulating.

In [ ]:
from symbulate import RV, Poisson

sims = RV(Poisson(3) * Poisson(2)).sim(4000)

plt.style.use(STYLE_PATH)  # re-apply: import + RVResults reset the style

arr = np.asarray(sims.results)
ax = plt.gca()
make_tile(arr[:, 0], arr[:, 1], ax)
plt.show()

## 5. Two variables with different numbers of distinct values

The tile grid does not need to be square. Here `X` takes 5 distinct values and
`Y` takes 10, so the plot is a 5-wide × 10-tall grid. `Y` grows with `X`
(`Y = 2X + a small step`), so the bright cells form a **diagonal staircase band**
climbing left to right — the joint pattern reads clearly even though the two axes
have different numbers of cells, and unobserved pairs stay dark. This is exactly
the discrete × discrete case where a single global bin count — as a 2D histogram
would use — makes no sense: the number of cells on each axis comes from that
variable's own distinct values.

In [ ]:
x5 = rng.integers(0, 5, 4000)            # 5 distinct values: 0, 1, 2, 3, 4
y10 = 2 * x5 + rng.integers(0, 2, 4000)  # 10 distinct values: 0..9, and Y grows with X

ax = plt.gca()
make_tile(x5, y10, ax)
plt.show()  # a 5-wide x 10-tall grid with a diagonal staircase band

## Verification checklist

- [ ] Section 1: one cell per distinct (x, y) integer pair; no binning
- [ ] Viridis colormap (not Blues); unobserved pairs are dark purple, frequent pairs yellow
- [ ] Colorbar on the right, sized proportionally to the axes
- [ ] Colorbar reads "Relative Frequency"; `normalize=False` switches it to "Count"
- [ ] Color scale starts at 0
- [ ] Each axis tick labeled with the distinct value; axis labels "X" and "Y"; title "Tile Plot"
- [ ] No grid fragments around the mesh
- [ ] Section 2: relative frequencies sum to 1; counts sum to the number of simulations
- [ ] Single tile plot prints no warning
- [ ] Second tile plot on the same axes draws AND prints the readability warning
- [ ] Works on real `.sim()` output, not just numpy arrays
- [ ] Section 5: non-square grid — X with 5 values, Y with 10 — renders as a 5-wide × 10-tall grid